In [ ]:
import os
import pandas as pd
from io import StringIO
import matplotlib.pyplot as plt

In [ ]:

# Chemins d'accès
CTD_PATH = r'Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Données brutes\CTD'
VUSITU_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Données brutes\TROLL"
BARO_PATH = r'Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx'
OLDDATA_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Themine_consolide_OLD.csv"
NEW_CONSOLIDATED_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Themines_consolide_updated.xlsx"


In [ ]:

# Fonction pour lire les fichiers ctd
def lire_fichier_CTD(filenum, path, skiprows=63):
    # Rechercher le fichier correspondant
    nom_fichier = [f for f in os.listdir(path) if f"Thémines_{filenum}" in f and f"Thémines_{filenum}" == f[:len(f"Thémines_{filenum}")]]
    if not nom_fichier:
        raise FileNotFoundError(f"Aucun fichier trouvé pour Thémines_{filenum} dans {path}")
    
    # Essayer d'abord de lire avec utf-8, puis fallback en latin1 si une erreur survient
    try:
        df = pd.read_csv(os.path.join(path, nom_fichier[0]), sep=';', encoding='utf-8', skiprows=skiprows)
    except UnicodeDecodeError:
        df = pd.read_csv(os.path.join(path, nom_fichier[0]), sep=';', encoding='latin1', skiprows=skiprows)
    
    # Supprimer la dernière ligne si elle est vide ou incorrecte
    df = df.iloc[:-1]
    
    # Conversion explicite des dates
    sample_date = df['Date/time'].iloc[0]
    if '/' in sample_date and sample_date[2] == '/':  
        df['Date/time'] = pd.to_datetime(df['Date/time'], dayfirst=True, errors='coerce')
    else: 
        df['Date/time'] = pd.to_datetime(df['Date/time'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
    
    # Arrondir les dates à l'heure la plus proche
    df['Date/time'] = df['Date/time'].dt.round('H')
    
    # Gestion de la conductivité : conversion en µS/cm si nécessaire
    if '2:Cond. spéc.[ms/cm]' in df.columns:
        # Appliquer la correction uniquement si le type est 'object' (chaîne de caractères)
        if df['2:Cond. spéc.[ms/cm]'].dtype == 'object':
            df['2:Cond. spéc.[ms/cm]'] = df['2:Cond. spéc.[ms/cm]'].str.replace(',', '.').str.strip()
        # Convertir en numérique et multiplier par 1000 pour µS/cm
        df['2:Cond. spéc.[ms/cm]'] = pd.to_numeric(df['2:Cond. spéc.[ms/cm]'], errors='coerce') * 1000
        df.rename(columns={'2:Cond. spéc.[ms/cm]': '2:Cond. spéc.[µS/cm]'}, inplace=True)
    
    # Remplacement des virgules par des points et conversion en numérique pour les autres colonnes
    for col in df.columns[1:]:  # Ignorer la colonne 'Date/time'
        if df[col].dtype == 'object':  # Vérifier si la colonne contient des chaînes de caractères
            df[col] = df[col].str.replace(',', '.').str.strip()  # Remplacer les virgules et enlever les espaces
        df[col] = pd.to_numeric(df[col], errors='coerce')  # Convertir en float en ignorant les erreurs
    
    return df

# Fonction pour lire et nettoyer un fichier VuSitu avec gestion de l'encodage
def lire_fichier_vusitu(filenum, path, skiprows=0):
    # Rechercher le fichier correspondant
    nom_fichier = [f for f in os.listdir(path) if f"VuSitu_{filenum}" in f]
    if not nom_fichier:
        raise FileNotFoundError(f"Aucun fichier trouvé pour VuSitu_{filenum} dans {path}")
    
    # Lecture du fichier avec tentative en utf-8, puis latin1 si échec
    file_path = os.path.join(path, nom_fichier[0])
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
    except UnicodeDecodeError:
        print(f"Échec de la lecture en utf-8 pour {nom_fichier[0]}, tentative avec latin1...")
        with open(file_path, 'r', encoding='latin1') as file:
            lines = file.readlines()
    
    # Enlever les guillemets redondants de chaque ligne
    cleaned_lines = ''.join([line.replace('"', '') for line in lines])

    # Utiliser StringIO pour lire les lignes nettoyées avec pandas
    df = pd.read_csv(StringIO(cleaned_lines), sep=',', skiprows=skiprows)

    # Conversion explicite de la colonne 'Date Heure' en datetime, puis arrondi à l'heure la plus proche
    if 'Date Heure' in df.columns:
        df['Date Heure'] = pd.to_datetime(df['Date Heure'], errors='coerce')
        df['Date Heure'] = df['Date Heure'].dt.round('H')  # Arrondir à l'heure la plus proche

    return df

# Fonction pour renommer les colonnes mal encodées
def renommer_colonnes(df):
    # Dictionnaire de renommage des colonnes incorrectement encodées
    colonnes_a_renommer = {
        'ConductivitÃ© spÃ©cifique (ÂµS/cm) (736870)': 'Conductivité spécifique (µS/cm) (736870)',
        'Concentration de chlorophylle-a (Âµg/L) (740874)': 'Concentration de chlorophylle-a (µg/L) (740874)',
        'TurbiditÃ© (NTU) (736551)': 'Turbidité (NTU) (736551)',
        'TempÃ©rature (Â°C) (740345)': 'Température (°C) (740345)'
    }
    
    # Renommer les colonnes si elles sont présentes dans le DataFrame
    df.rename(columns=colonnes_a_renommer, inplace=True)
    
    return df


In [ ]:

# Lire les données barométriques à partir du fichier Excel
baro_data = pd.read_excel(BARO_PATH)

# Initialiser un DataFrame vide pour stocker les données fusionnées
merge_ctd_df = pd.DataFrame()

# Boucle pour traiter les fichiers CTD
for num in range(1, 46+1):
    filenum = str(num)  # Convertir le numéro en chaîne

    # Lire les données CTD
    CTD = lire_fichier_CTD(filenum, CTD_PATH, skiprows=63)

    # Fusionner avec les données Baro sur la base des dates
    merged_df = pd.merge(CTD, baro_data[['DATE', 'Patm Thémines [hPa]']], left_on='Date/time', right_on='DATE', how='left')

    # Calcul de la hauteur piézométrique : correction par soustraction de la pression barométrique
    # On suppose que la pression CTD est en cmH2O et barométrique en hPa (conversion nécessaire)
    merged_df['Niveau_(cm)'] = merged_df['Pression[cmH2O]'] - merged_df['Patm Thémines [hPa]']
    
    # Renommer la colonne de conductivité et température pour les rendre plus claires
    merged_df.rename(columns={'2:Cond. spéc.[µS/cm]': 'Cond_(µS/cm)', 'Température[°C]': 'Temp_(°C)'}, inplace=True)

    # Conserver les colonnes 'Date/time', 'Niveau_(cm)', 'Pression[cmH2O]' (CTD), 'Patm Thémines [hPa]', 'Cond_(µS/cm)', 'Temp_(°C)'
    merge_ctd_df = pd.concat([merge_ctd_df, merged_df[['Date/time', 'Niveau_(cm)', 'Pression[cmH2O]', 'Patm Thémines [hPa]', 'Cond_(µS/cm)', 'Temp_(°C)']]], ignore_index=True)

# Afficher un aperçu des premières lignes du DataFrame final
merge_ctd_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Correction de la lecture des anciennes données
olddatata_df = pd.read_csv(OLDDATA_PATH, sep=';')
olddatata_df['DATE'] = pd.to_datetime(olddatata_df['DATE'], dayfirst=True)

# Conversion des dates des nouvelles données CTD
merge_ctd_df['DATE'] = pd.to_datetime(merge_ctd_df['Date/time'])

# Step 2: Identifier les dates de jonction
last_date_old = olddatata_df['DATE'].max()  # Dernière date dans l'ancien fichier
first_date_new = merge_ctd_df['DATE'].min()  # Première date dans le nouveau fichier

print(f"Dernière date dans les anciennes données : {last_date_old}")
print(f"Première date dans les nouvelles données : {first_date_new}")

# Step 3: Filtrer les données autour de la jonction (7 jours avant et après)
old_data_filtered = olddatata_df[olddatata_df['DATE'] >= (last_date_old - pd.Timedelta(days=7))]
new_data_filtered = merge_ctd_df[merge_ctd_df['DATE'] <= (first_date_new + pd.Timedelta(days=7))]

# Step 4: Calculer le décalage pour le niveau d'eau (Niveau)
dernier_niveau_old = old_data_filtered['Niveau'].dropna().iloc[-1]  # Dernier niveau valide dans les anciennes données
premier_niveau_new = new_data_filtered['Niveau_(cm)'].dropna().iloc[0]  # Premier niveau valide dans les nouvelles données
shift_niveau = dernier_niveau_old - premier_niveau_new  # Calculer le décalage

# Appliquer le décalage aux nouvelles données
merge_ctd_df['Niveau_corr'] = merge_ctd_df['Niveau_(cm)'] + shift_niveau

# Step 5: Calculer le décalage pour la conductivité (Conducti)
derniere_cond_old = old_data_filtered['Conducti'].dropna().iloc[-1]  # Dernière conductivité valide dans les anciennes données
premiere_cond_new = new_data_filtered['Cond_(µS/cm)'].dropna().iloc[0]  # Première conductivité valide dans les nouvelles données
shift_cond = derniere_cond_old - premiere_cond_new  # Calculer le décalage pour la conductivité

# Appliquer le décalage aux nouvelles données pour la conductivité
merge_ctd_df['Conducti_corr'] = merge_ctd_df['Cond_(µS/cm)'] + shift_cond

# Graphique pour contrôler la correction 
new_data_filtered_corr = merge_ctd_df[merge_ctd_df['DATE'] <= (first_date_new + pd.Timedelta(days=7))]
fig, ax1 = plt.subplots(figsize=(11, 3))
ax1.plot(old_data_filtered['DATE'], old_data_filtered['Niveau'], label='Niveau (Consolidé)', color='green')
ax1.plot(new_data_filtered_corr['DATE'], new_data_filtered_corr['Niveau_corr'], label='Niveau (Nouveau - Corrigé)', color='blue')
ax1.set_ylabel('Niveau (cm)')
ax1.tick_params(axis='y')

ax2 = ax1.twinx()
ax2.plot(old_data_filtered['DATE'], old_data_filtered['Conducti'], label='Conductivité (Consolidé)', color='orange')
ax2.plot(new_data_filtered_corr['DATE'], new_data_filtered_corr['Conducti_corr'], label='Conductivité (Nouveau - Corrigé)', color='red')
ax2.set_ylabel('Conductivité (µS/cm)')
ax2.tick_params(axis='y')

fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4)
plt.tight_layout()
plt.show()


In [ ]:
# Boucle pour lire plusieurs fichiers VuSitu et les concaténer
merge_troll_df = pd.DataFrame()
for num in range(1,40+1):
    try:
        df_vusitu = lire_fichier_vusitu(num, VUSITU_PATH)
        
        # Renommer les colonnes mal encodées
        df_vusitu = renommer_colonnes(df_vusitu)
        
        merge_troll_df = pd.concat([merge_troll_df, df_vusitu], ignore_index=True)
    except FileNotFoundError as e:
        print(e)

merge_troll_df

In [ ]:
olddatata_df = pd.read_csv(OLDDATA_PATH, sep=';')
olddatata_df['DATE'] = pd.to_datetime(olddatata_df['DATE'], dayfirst=True)

niveau_ngf_Thémines = 311.261

# Renommer 'Date Heure' en 'DATE' pour les données VuSitu (Aquatroll)
merge_troll_df.rename(columns={'Date Heure': 'DATE'}, inplace=True)
merge_ctd_df.rename(columns={'Date/time': 'DATE'}, inplace=True)
# Conversion des dates en datetime pour l'ensemble des DataFrames
olddatata_df['DATE'] = pd.to_datetime(olddatata_df['DATE'], errors='coerce')
merge_ctd_df['DATE'] = pd.to_datetime(merge_ctd_df['DATE'], errors='coerce')
merge_troll_df['DATE'] = pd.to_datetime(merge_troll_df['DATE'], errors='coerce')

# Renommer les colonnes des nouvelles données CTD pour correspondre au format attendu
merge_ctd_df_renamed = merge_ctd_df[['DATE', 'Niveau_(cm)', 'Cond_(µS/cm)', 'Temp_(°C)']].rename(
    columns={'Niveau_(cm)': 'Niveau', 'Cond_(µS/cm)': 'Conducti', 'Temp_(°C)': 'Temp'})

# Renommer les colonnes des données Aquatroll (VuSitu)
merge_troll_df_renamed = merge_troll_df.rename(columns={
    'Concentration RDO (mg/L) (735995)': 'O2',
    'Conductivité spécifique (µS/cm) (736870)': 'Xtroll',
    'Fluorescence de chlorophylle-a (RFU) (740874)': 'FluoChloro_a',
    'Concentration de chlorophylle-a (µg/L) (740874)': 'ConChloro_a',
    'Turbidité (NTU) (736551)': 'Turbidity',
    'Température (°C) (740345)': 'TempTroll',
    'Saturation RDO (%Sat) (735995)': 'O2 (%Sat)'
})

# Fusionner les données CTD et Aquatroll (VuSitu) sur la base des dates
merged_new_data = pd.merge(merge_ctd_df_renamed, merge_troll_df_renamed, on='DATE', how='outer')

# Calcul de 'Niveau_NGF' : à partir de 'Niveau' selon la formule donnée
merged_new_data['Niveau_NGF'] = niveau_ngf_Thémines - (niveau_ngf_Thémines - merged_new_data['Niveau']) / 100

# Calcul de 'Q' (Débit) en fonction des courbes de tarage
Q_bas = merged_new_data['Niveau'] < 21.4  
Q_haut = merged_new_data['Niveau'] >= 21.4

# Courbe de tarage pour Niveau < 21.4 cm
merged_new_data.loc[Q_bas, 'Q'] = 50.485 * (merged_new_data.loc[Q_bas, 'Niveau'] / 100) + 1

# Courbe de tarage pour Niveau >= 21.4 cm
merged_new_data.loc[Q_haut, 'Q'] = 8325.3 * (merged_new_data.loc[Q_haut, 'Niveau'] / 100) ** 2 - 2266.1 * (merged_new_data.loc[Q_haut, 'Niveau'] / 100) + 116.21

# Concaténer les anciennes données avec les nouvelles données fusionnées
full_data = pd.concat([olddatata_df, merged_new_data], ignore_index=True)

# Trier les données par date pour garantir un ordre chronologique
full_data = full_data.sort_values(by='DATE').reset_index(drop=True)

full_data

In [ ]:
# Sauvegarder les nouvelles données consolidées dans un fichier Excel
full_data.to_excel(NEW_CONSOLIDATED_PATH, index=False)
print(f"Nouvelles données consolidées enregistrées sous : {NEW_CONSOLIDATED_PATH}")

In [ ]:
# Lire le fichier Excel consolidé
data = pd.read_excel(NEW_CONSOLIDATED_PATH)

# Plot Conductivity for both CTD and Aquatroll (VuSitu)
plt.figure(figsize=(10, 6))

# Plot for CTD Conductivity
plt.plot(data['DATE'], data['Conducti'], label='Conductivité CTD(µS/cm)', color='blue', alpha=0.5)

# Plot for Aquatroll (VuSitu) Conductivity
plt.plot(data['DATE'], data['Xtroll'], label=' Conductivité Aquatroll (µS/cm)', color='orange',alpha=0.5)

# Set labels and title
plt.xlabel('Date')
plt.ylabel('Conductivity (µS/cm)')
plt.title('Conductivity of CTD and Aquatroll over Time')
plt.legend()
plt.grid(True)

# Show the plot
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Path to the newly saved consolidated data
new_consolidated_path = os.path.join(NEW_CONSOLIDATED_PATH)
punctual_data_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\punctual_measurements.xlsx")

# Load the new consolidated data
new_consolidated_df = pd.read_excel(new_consolidated_path)
punctual_df = pd.read_excel(punctual_data_path)

# Ensure 'DATE' column in consolidated data and 'Jour' column in punctual data are in datetime format
new_consolidated_df['DATE'] = pd.to_datetime(new_consolidated_df['DATE'])
punctual_df['Datetime'] = pd.to_datetime(punctual_df['Jour'], dayfirst=True, errors='coerce')

# Plot the consolidated data (Niveau and Conductivity) as lines and the punctual data as points
fig, ax1 = plt.subplots(figsize=(11, 5))

# Plot punctual measurements for water level as points (control points)
ax1.scatter(punctual_df['Datetime'], punctual_df['Hauteur (cm)'], color='blue', label='Points de contrôle', marker='x')
# Plot water level (Niveau) from consolidated data as a line
ax1.plot(new_consolidated_df['DATE'], new_consolidated_df['Niveau'], label='Niveau', color='grey', alpha =0.5)
ax1.set_ylabel('Niveau (cm)')
ax1.set_xlabel('Datetime')
ax1.tick_params(axis='y')


# Add legends and adjust layout
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4)
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import os

# Path to the newly saved consolidated data
new_consolidated_path = os.path.join(NEW_CONSOLIDATED_PATH)
punctual_data_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\punctual_measurements.xlsx")

# Load the new consolidated data
new_consolidated_df = pd.read_excel(new_consolidated_path)
punctual_df = pd.read_excel(punctual_data_path)

# Ensure 'DATE' column in consolidated data and 'Jour' column in punctual data are in datetime format
new_consolidated_df['DATE'] = pd.to_datetime(new_consolidated_df['DATE'])
punctual_df['Datetime'] = pd.to_datetime(punctual_df['Jour'], dayfirst=True, errors='coerce')

# Create a Plotly figure
fig = go.Figure()

# Add scatter points for punctual measurements (control points)
fig.add_trace(go.Scatter(
    x=punctual_df['Datetime'],
    y=punctual_df['Hauteur (cm)'],
    mode='markers',
    name='Points de contrôle',
    marker=dict(color='blue', symbol='x')
))

# Add line for consolidated water level (Niveau)
fig.add_trace(go.Scatter(
    x=new_consolidated_df['DATE'],
    y=new_consolidated_df['Niveau'],
    mode='lines',
    name='Niveau_(cm)',
    line=dict(color='grey', width=2, dash='solid')
))

# Customize layout
fig.update_layout(
    xaxis_title='Datetime',
    yaxis_title='Niveau (cm)',
    legend_title='Légende',
    template='plotly_white'
)

# Show the figure
fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import os

# Chemins vers les fichiers
punctual_data_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\punctual_measurements_Niveau.xlsx")
new_consolidated_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Themines_consolide_updated.xlsx")

# Charger les données
new_consolidated_df = pd.read_excel(new_consolidated_path)
punctual_df = pd.read_excel(punctual_data_path)

# Conversion des dates
new_consolidated_df['DATE'] = pd.to_datetime(new_consolidated_df['DATE'], errors='coerce')
punctual_df['Datetime'] = pd.to_datetime(punctual_df['Jour'], dayfirst=True, errors='coerce')

# Dates des valeurs aberrantes à supprimer
outlier_dates = [pd.to_datetime("2024-03-07 11:00"),
                 pd.to_datetime("2020-01-21 12:00")
                ]
    
# Supprimer les valeurs aberrantes
new_consolidated_df = new_consolidated_df[~new_consolidated_df['DATE'].isin(outlier_dates)]

# Interpolation pour combler les valeurs manquantes créées par la suppression
new_consolidated_df.set_index('DATE', inplace=True)  # Mettre 'DATE' comme index pour une interpolation correcte
new_consolidated_df['Niveau'] = new_consolidated_df['Niveau'].interpolate(method='time')  # Interpolation temporelle
new_consolidated_df.reset_index(inplace=True)

# Appliquer les corrections uniquement pour les points marqués "Oui"
for index, row in punctual_df.iterrows():
    point_date = row['Datetime']
    point_value = row['Hauteur (cm)']
    apply_correction = row.get("Correction")  # Vérifier la colonne "Correction"

    if pd.isna(point_value) or apply_correction != "Oui":
        continue  # Passer les points sans correction ou avec valeur NaN

    # Trouver la mesure la plus proche dans les données consolidées
    nearest_measure_index = (new_consolidated_df['DATE'] - point_date).abs().idxmin()
    nearest_measure_value = new_consolidated_df.at[nearest_measure_index, 'Niveau']

    if pd.isna(nearest_measure_value) or abs((new_consolidated_df['DATE'][nearest_measure_index] - point_date).total_seconds()) > 3600:
        continue

    # Calculer le décalage et l'appliquer
    decalage = point_value - nearest_measure_value
    print(f"Date : {point_date}, Décalage appliqué : {decalage} cm")

    # Appliquer le décalage à toutes les mesures après la date du point de contrôle
    new_consolidated_df.loc[new_consolidated_df['DATE'] >= point_date, 'Niveau'] += decalage

# Calcul de 'Q' (Débit) en fonction des courbes de tarage
Q_bas = new_consolidated_df['Niveau'] < 21.4  
Q_haut = new_consolidated_df['Niveau'] >= 21.4

# Courbe de tarage pour Niveau < 21.4 cm
new_consolidated_df.loc[Q_bas, 'Q'] = 50.485 * (new_consolidated_df.loc[Q_bas, 'Niveau'] / 100) + 1

# Courbe de tarage pour Niveau >= 21.4 cm
new_consolidated_df.loc[Q_haut, 'Q'] = (
    8325.3 * (new_consolidated_df.loc[Q_haut, 'Niveau'] / 100) ** 2
    - 2266.1 * (new_consolidated_df.loc[Q_haut, 'Niveau'] / 100)
    + 116.21
)

# Visualisation interactive avec Plotly
fig = go.Figure()

# Tracer les données corrigées pour Niveau et Q
fig.add_trace(go.Scatter(
    x=new_consolidated_df['DATE'], y=new_consolidated_df['Niveau'],
    mode='lines', name='Niveau corrigé', line=dict(color='green')
))

# Ajouter les points de contrôle avec indication de correction (Rouge si corrigé, Bleu sinon)
corrected_color = ['red' if val == "Oui" else 'blue' for val in punctual_df['Correction']]
fig.add_trace(go.Scatter(
    x=punctual_df['Datetime'], y=punctual_df['Hauteur (cm)'],
    mode='markers', name='Points de contrôle', marker=dict(color=corrected_color, symbol='x', size=8)
))

# Configurer le graphique
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Niveau (cm)",
    legend_title="Données"
)

# Exporter en HTML pour visualisation interactive
fig.write_html("Niveau_Q_Corrige.html")
fig.show()

# Sauvegarder les données corrigées dans un fichier Excel
output_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Thémines_Niveau_corrected.xlsx")
new_consolidated_df.to_excel(output_path, index=False)
print(f"Données corrigées et débit mis à jour sauvegardés dans : {output_path}")


In [ ]:
new_consolidated_path = os.path.join(r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Thémines_Niveau_corrected.xlsx")
new_consolidated_df = pd.read_excel(new_consolidated_path)

# Plot the consolidated data (Conductivity) as a line and the punctual data as points
fig, ax = plt.subplots(figsize=(11, 5))

# Plot conductivity from consolidated data as a line
ax.plot(new_consolidated_df['DATE'], new_consolidated_df['Xtroll'], label='Conductivité CTD ', color='orange', alpha=0.5)

# Plot punctual measurements for conductivity as points (control points)
ax.scatter(punctual_df['Datetime'], punctual_df['Conductivité'], color='red', label='Points de contrôle', marker='x')

# Set labels and titles
ax.set_ylabel('Conductivité (µS/cm)')
ax.set_xlabel('Datetime')

# Add legends and adjust layout
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2)
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import os

# Chemin vers le fichier de données
data_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Thémines_Niveau_corrected.xlsx"
)
punctual_data_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\punctual_measurements_Brute.xlsx"
)

# Charger les données
data = pd.read_excel(data_path)
punctual_df = pd.read_excel(punctual_data_path)

# Définir les périodes et les colonnes correspondantes pour suppression
periods_to_remove = [
    ("2024-07-31 11:00", "2024-09-18 16:00", "Conducti"),
    ("2019-02-27 17:00", "2019-03-30 11:00", "Conducti")

]

# Convertir 'DATE' en format datetime pour manipulation
data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')

# Appliquer les suppressions en fonction des périodes définies
for period in periods_to_remove:
    start, end, column = pd.to_datetime(period[0], errors='coerce'), pd.to_datetime(period[1], errors='coerce'), period[2]
    if start > end:  # Corriger les inversions de dates si nécessaire
        start, end = end, start
    data.loc[(data['DATE'] >= start) & (data['DATE'] <= end), column] = None
# Supprimer les valeurs aberrantes
data = data[~data['DATE'].isin(outlier_dates)]

# Conversion des dates en datetime
data['DATE'] = pd.to_datetime(data['DATE'], errors='coerce')
punctual_df['Datetime'] = pd.to_datetime(punctual_df['Jour'], dayfirst=True, errors='coerce')

# Créer la colonne "Conductivité" avec comblement des données manquantes en priorisant Conducti
data['Conductivité'] = None
decalage = 0

# Logique de comblement avec priorité sur Conducti
for i in range(len(data)):
    if not pd.isna(data.at[i, 'Conducti']):  # Utiliser Conducti en priorité
        data.at[i, 'Conductivité'] = data.at[i, 'Conducti']
        decalage = data.at[i, 'Conducti'] - data.at[i, 'Xtroll']  # Calculer le décalage
    elif not pd.isna(data.at[i, 'Xtroll']):  # Utiliser Xtroll avec décalage si Conducti est manquant
        data.at[i, 'Conductivité'] = data.at[i, 'Xtroll'] + decalage

# Visualisation interactive pour choisir les points de contrôle
fig = go.Figure()

# Ajouter les lignes pour Xtroll, Conducti et Conductivité
fig.add_trace(go.Scatter(x=data['DATE'], y=data['Xtroll'], mode='lines', name='Xtroll', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=data['DATE'], y=data['Conducti'], mode='lines', name='Conducti', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=data['DATE'], y=data['Conductivité'], mode='lines', name='Conductivité', line=dict(color='green')))

# Ajouter les points de contrôle
fig.add_trace(go.Scatter(
    x=punctual_df['Datetime'],
    y=punctual_df['Conductivité'],
    mode='markers',
    name='Points de contrôle',
    marker=dict(symbol='x', color='red', size=8)
))

# Configuration du graphique
fig.update_layout(
    title='Visualisation des données de conductivité avec comblement et points de contrôle',
    xaxis_title='Date',
    yaxis_title='Conductivité (µS/cm)',
    legend_title='Type de mesure'
)

# Exporter le graphique pour inspection
html_output_path = (r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Graphique_conductivté et points_de_controles.html")
fig.write_html(html_output_path)
print(f"Graphique interactif sauvegardé dans : {html_output_path}")

output_path = os.path.join(
    r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Thémines_Calé_Nettoyé.xlsx"
)
data.to_excel(output_path, index=False)
print(f"Données corrigées et interpolées sauvegardées dans : {output_path}")


# Afficher le graphique
fig.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Chemins des fichiers de données
rainfall_data_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan\Données brutes\Pluie_BV_Ouysse.csv"
hydro_data_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Thémines_Calé_Nettoyé.xlsx"

# Chargement des données
rainfall_data = pd.read_csv(rainfall_data_path)
hydro_data = pd.read_excel(hydro_data_path)

# Conversion des colonnes de dates en datetime
rainfall_data['Date'] = pd.to_datetime(rainfall_data['Date'], errors='coerce')
hydro_data['DATE'] = pd.to_datetime(hydro_data['DATE'], errors='coerce')

# Moyenne mobile pour les paramètres
hydro_data['Temp_Moving_Avg'] = hydro_data['Temp'].rolling(window=24, center=True).mean()
hydro_data['Q'] = hydro_data['Q']/1000
hydro_data['O2_Moving_Avg'] = hydro_data['O2'].rolling(window=24, center=True).mean()
hydro_data['FluoChloro_a_Moving_Avg'] = hydro_data['FluoChloro_a'].rolling(window=24, center=True).mean()
hydro_data['Turbidity_Moving_Avg'] = hydro_data['Turbidity'].rolling(window=24, center=True).mean()

# Ajustement des dates pour correspondre à la plage du niveau d'eau avec marge de 15 jours
start_date = hydro_data['DATE'].min() - pd.Timedelta(days=15)
end_date = hydro_data['DATE'].max() + pd.Timedelta(days=15)
rainfall_data = rainfall_data[(rainfall_data['Date'] >= start_date) & (rainfall_data['Date'] <= end_date)]
hydro_data = hydro_data[(hydro_data['DATE'] >= start_date) & (hydro_data['DATE'] <= end_date)]

# Création des graphiques
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={'hspace': 0.05})

# --- Graphe 1 : Niveau d'eau et Précipitations ---
ax1 = axes[0]
ax2 = ax1.twinx()
ax2.bar(
    rainfall_data['Date'], rainfall_data['Precipitation (mm)'], 
    width=0.8, color='royalblue', align='center'
)
ax2.invert_yaxis()
ax2.set_ylabel('Précipitations (mm)', color='royalblue')
ax2.tick_params(axis='y', labelcolor='royalblue')

ax1.plot(hydro_data['DATE'], hydro_data['Q'], color='lightseagreen')
ax1.set_ylabel('Débit (m³/s)', color='lightseagreen')
ax1.tick_params(axis='y', labelcolor='lightseagreen')
ax1.set_xlim([start_date, end_date]) 

# --- Graphe 2 : Conductivité et Température ---
ax3 = axes[1]
ax4 = ax3.twinx()
ax3.plot(hydro_data['DATE'], hydro_data['Conductivité'], color='black')
ax3.set_ylabel('Conductivité (µS/cm)', color='black')
ax3.tick_params(axis='y', labelcolor='black')

ax4.plot(hydro_data['DATE'], hydro_data['Temp_Moving_Avg'], color='crimson')
ax4.set_ylabel('Température (°C)', color='crimson')
ax4.tick_params(axis='y', labelcolor='crimson')
ax3.set_xlim([start_date, end_date])  # Définir la plage x

# --- Graphe 3 : Turbidité, Oxygène et Chlorophylle (Moyenne Mobile) ---
ax5 = axes[2]
ax6 = ax5.twinx()
ax5.plot(hydro_data['DATE'], hydro_data['Turbidity_Moving_Avg'], color='darkorange')
ax5.set_ylabel('Turbidité (NTU)', color='darkorange')
ax5.tick_params(axis='y', labelcolor='darkorange')
ax5.set_ylim(0, 200)

# Ajouter un troisième axe pour la chlorophylle
ax7 = ax5.twinx()
ax7.spines['right'].set_position(('outward', 40))  # Décaler le troisième axe
ax7.plot(hydro_data['DATE'], hydro_data['FluoChloro_a_Moving_Avg'], color='green')
ax7.set_ylabel('Chlorophylle (RFU)', color='green')
ax7.tick_params(axis='y', labelcolor='green')

ax6.plot(hydro_data['DATE'], hydro_data['O2_Moving_Avg'], color='darkmagenta')
ax6.set_ylabel('Oxygène (mg/L)', color='darkmagenta')
ax6.tick_params(axis='y', labelcolor='darkmagenta')
ax5.set_xlim([start_date, end_date])  # Définir la plage x

# Ajustement des marges et sauvegarde en SVG
plt.subplots_adjust(hspace=0.15)  # Réduire l'espace entre les graphes
svg_output_path = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Thémines\Gaetan\Graphes.svg"
plt.savefig(svg_output_path, format='svg', dpi=300)
print(f"Graphique sauvegardé en SVG à : {svg_output_path}")

plt.show()


In [ ]:
# Import the necessary library
import matplotlib.pyplot as plt

# Setting up the figure size and subplots grid (3 rows, 2 columns)
fig, axs = plt.subplots(3, 2, figsize=(14, 10))

# Plot for 'Niveau' and 'Q' on two axes (Niveau on the left y-axis, Q on the right y-axis)
ax1 = axs[0, 0]  # First subplot
ax2 = ax1.twinx()  # Create a second y-axis on the same subplot

# Plot Niveau (blue line)
ax1.plot(data['DATE'], data['Niveau'], label='Niveau (cm)', color='blue')
# Plot Débit (Q) (red line)
ax2.plot(data['DATE'], data['Q'], label='Débit (Q)', color='red')

# Set axis labels and title
ax1.set_xlabel('Date')
ax1.set_ylabel('Niveau (cm)', color='blue')
ax2.set_ylabel('Débit (l/s)', color='red')
ax1.set_title('Niveau et Débit')

# Show legends for both y-axes
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

# Plot for 'Temp' (CTD) and 'TempTroll' (Aquatroll) over time
axs[0, 1].plot(data['DATE'], data['Temp'], label='CTD', color='orange')
axs[0, 1].plot(data['DATE'], data['TempTroll'], label='Aquatroll', color='purple')
axs[0, 1].set_title('Temperature (CTD & Aquatroll)')
axs[0, 1].set_xlabel('Date')
axs[0, 1].set_ylabel('Temperature (°C)')
axs[0, 1].legend()

# Plot for 'Conductivity' (CTD) and 'Xtroll' (Aquatroll) over time
axs[1, 0].plot(data['DATE'], data['Conducti'], label='CTD', color='blue')
axs[1, 0].plot(data['DATE'], data['Xtroll'], label='Aquatroll', color='green')
axs[1, 0].set_title('Conductivité (CTD & Aquatroll)')
axs[1, 0].set_xlabel('Date')
axs[1, 0].set_ylabel('Conductivité (µS/cm)')
axs[1, 0].legend()

# Separate plot for 'Turbidity'
axs[1, 1].plot(data['DATE'], data['Turbidity'], label='Turbidité', color='cyan')
axs[1, 1].set_title('Turbidité')
axs[1, 1].set_xlabel('Date')
axs[1, 1].set_ylabel('Turbidity')
axs[1, 1].legend()

# Separate plot for 'FluoChloro_a'
axs[2, 0].plot(data['DATE'], data['FluoChloro_a'], label='FluoChloro_a', color='yellow')
axs[2, 0].set_title('Fluorescence de chlorophylle-a')
axs[2, 0].set_xlabel('Date')
axs[2, 0].set_ylabel('FluoChloro_a')
axs[2, 0].legend()

# Turn off the empty subplot (bottom-right)
axs[2, 1].axis('off')

# Adjust layout to prevent overlapping of titles, labels, and legends
plt.tight_layout()

# Show the plot
plt.show()
